# Example: Let's Compute and Analyze the Covariance Matrix for our Dataset
In this example, we will compute the covariance matrix for our dataset of equity growth rates, and then analyze some interesting entries in the covariance matrix.

> __Learning Objectives:__
>
> By the end of this example, you should be able to:
> * __Compute empirical covariance matrices__: Calculate covariance matrices from historical growth-rate data using matrix operations and verify results against built-in functions.
> * __Analyze covariance relationships__: Examine covariance and correlation between specific asset pairs and interpret their relationships through data visualization.
> * __Understand covariance implications__: Discuss the meaning of positive and negative covariances for portfolio construction and risk management.

This is going to be interesting, so let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's set up our code environment:

In [3]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/) and the [CHEME 5660 Quantitative Finance Package documentation](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/).

### Data
We gathered daily open-high-low-close (OHLC) data for each firm in the [S&P500](https://en.wikipedia.org/wiki/S%26P_500) from `01-03-2014` until `12-31-2024`, along with data for several exchange-traded funds and volatility products during that time period. 

Let's load the `original_dataset::DataFrame` by calling [the `MyTrainingMarketDataSet()` function](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/data/#VLQuantitativeFinancePackage.MyTrainingMarketDataSet) and remove firms that do not have the maximum number of trading days. The cleaned dataset $\mathcal{D}$ will be stored in the `dataset` variable.

In [6]:
original_dataset = MyTrainingMarketDataSet() |> x-> x["dataset"];

Not all tickers in our dataset have the maximum number of trading days for various reasons, such as acquisition or delisting events. Let's collect only those tickers with the maximum number of trading days.

First, let's compute the number of records for a firm that we know has the maximum value, e.g., `AAPL`, and save that value in the `maximum_number_trading_days::Int64` variable:

In [8]:
maximum_number_trading_days = original_dataset["AAPL"] |> nrow;

Now, let's iterate through our data and collect only tickers with `maximum_number_trading_days` records. Save that data in the `dataset::Dict{String,DataFrame}` variable:

In [10]:
dataset = let

    # initialize -
    dataset = Dict{String, DataFrame}();

    # iterate through the dictionary; we can't guarantee a particular order
    for (ticker, data) ∈ original_dataset  # we get each (K, V) pair!
        if (nrow(data) == maximum_number_trading_days) # check if ticker has maximum trading days
            dataset[ticker] = data;
        end
    end
    dataset; # return
end;

Next, let's get a list of the firms in our cleaned dataset and sort them alphabetically. We store the sorted firm ticker symbols in the `list_of_tickers::Array{String,1}` variable:

In [12]:
list_of_tickers = keys(dataset) |> collect |> sort; # list of firm "ticker" symbols in alphabetical order

Now, let's load the GBM parameters that we computed in the previous example:

In [14]:
parameters_df = let

    # load -
    df = CSV.read(joinpath(_PATH_TO_DATA,"SAGBM-Parameters-Fall-2025.csv"), DataFrame);
    df; # return
end;

__Reminder:__ What's in the `parameters_df::DataFrame` variable?

In [16]:
parameters_df

Row,ticker,drift,t,lower_bound_drift,upper_bound_drift,volatility
,String7,Float64,Float64,Float64,Float64,Float64
1,A,0.147468,1.96,0.145161,0.149774,0.231426
2,AAL,-0.141294,1.96,-0.1436,-0.138988,0.454992
3,AAP,-0.0481021,1.96,-0.0504085,-0.0457958,0.342562
4,AAPL,0.24271,1.96,0.240403,0.245016,0.234474
5,ABBV,0.116,1.96,0.113693,0.118306,0.242938
6,ABT,0.123083,1.96,0.120777,0.125389,0.199723
7,ACN,0.151012,1.96,0.148706,0.153319,0.214367
8,ADBE,0.224401,1.96,0.222094,0.226707,0.276273
9,ADI,0.146449,1.96,0.144142,0.148755,0.259329


Finally, let's specify some constants that we will use in our analysis. Check out the comments for details on the constants, their permissible values, units, etc.

In [18]:
T = 252; # number of trading days in a year
Δt = (1/T); # time step (1 trading day, in years)

___

## Task 1: Compute the Empirical Covariance Matrix
In this task, let's compute the empirical growth-rate covariance matrix $\hat{\mathbf{\Sigma}}_g$ for our dataset $\mathcal{D}$ using code that we write ourselves (we'll never do this in practice, but it's a good exercise). The empirical covariance matrix is given by:
$$
\hat{\mathbf{\Sigma}}_g = \frac{1}{n-1}\tilde{\mathbf{X}}^{\top}\tilde{\mathbf{X}}
$$
where $\tilde{\mathbf{X}}$ is the centered data matrix:
$$
\tilde{\mathbf{X}} = \mathbf{X} - \mathbf{1}\mathbf{m}^{\top}
$$
where $\mathbf{1} \in \mathbb{R}^{n}$ is a vector of ones, and $\mathbf{1}\mathbf{m}^{\top}$ creates an $n \times m$ matrix where each row is identical and contains the mean growth rates on the columns. 

> __Outer product:__ The $\mathbf{1}\mathbf{m}^{\top}$ is an example of an outer product. The [outer product](https://en.wikipedia.org/wiki/Outer_product) of two vectors $\mathbf{a} \in \mathbb{R}^{n}$ and $\mathbf{b} \in \mathbb{R}^{m}$ is the $n \times m$ matrix $\mathbf{a}\mathbf{b}^{\top}$. Each element of the outer product is computed as $(\mathbf{a}\mathbf{b}^{\top})_{ij} = a_i b_j$. 

Let's construct the growth-rate matrix $\mathbf{X} \in\mathbb{R}^{n \times m}$, where each row contains the growth rates for all $m$ firms at one time step, using `log_growth_matrix(...)`. The associated log return is $r_k^{(i)}=\Delta t\,g_k^{(i)}$, but this analysis remains in growth-rate units. 

We store the growth-rate data in the `X::Array{Float64,2}` variable:

In [ ]:
X = log_growth_matrix(dataset, list_of_tickers) # growth rates g

Next, let's compute the mean growth rate for each firm and store it in the `m::Array{Float64,1}` variable:

In [ ]:
m = mean(X, dims=1) |> vec # mean growth rate for each firm

Now, let's form the centered data matrix $\tilde{\mathbf{X}}$ by subtracting the mean growth rates from each row of $\mathbf{X}$. We store the centered data in `X_centered::Array{Float64,2}`:

In [ ]:
r, c = size(X)
ones_vector = ones(r)
⊗(ones_vector, m) # outer product of ones_vector and m

In [ ]:
X_centered = let 
    r, c = size(X)
    ones_vector = ones(r)
    X̃ = X .- ⊗(ones_vector, m);
end

Finally, let's compute the empirical growth-rate covariance matrix $\hat{\mathbf{\Sigma}}_g$ and store it in the

In [ ]:
Σ̂_g, Ĉ = let
   (n, m) = size(X_centered)
   Σ_g = (1/(n-1)) * (X_centered' * X_centered); # Cov(g), units: 1/year²
   C = Δt*Σ_g; # GBM covariance rate, units: 1/year
   (Σ_g, C)
end

### Visualize the Covariance Matrix
Let's create a heatmap of the covariance matrix to visualize the relationships between assets in the subset at once. This provides a quick overview of which assets are highly correlated (bright colors) versus uncorrelated or negatively correlated (darker colors).

In [ ]:
let
    # select a subset of tickers for better visualization (up to 40)
    n_show = min(40, length(list_of_tickers));
    subset_Σ̂_g = Σ̂_g[1:n_show, 1:n_show];
    
    # create heatmap
    heatmap(subset_Σ̂_g, 
        xticks=false, 
        yticks=false, 
        axis=false,
        title="Covariance Matrix Heatmap (Subset)", 
        color=:cividis,
        aspect_ratio=:equal,
        yflip=true,
        colorbar_fontsize=8)
end

### Check: How does our answer compare to the built-in `cov(...)` function?
We should almost always buy versus build, i.e., use built-in functions versus writing our own. Let's compare our answer to [the built-in `cov(...)` function](https://docs.julialang.org/en/v1/stdlib/Statistics/#Statistics.cov).

In [ ]:
Σ_g_builtin = let
    G = log_growth_matrix(dataset, list_of_tickers); # growth-rate matrix
    cov(G, dims=1); # Cov(g), no Δt scaling
end

@assert isapprox(Σ̂_g, Σ_g_builtin, rtol=1e-10)

### Check: Can we independently verify that our covariance matrix is approximately correct?
Yes! There are two(ish) ways that we can do this.

First, we can check that the diagonal elements of the covariance rate $\hat C$ are approximately equal to the square of the volatility estimates that we computed in the previous examples. 

In [ ]:
let
    # initialize -
    ticker_to_check = "AMD";
    i = findfirst(x-> x==ticker_to_check, list_of_tickers);
    σ_ii = sqrt(Ĉ[i,i]); # estimated volatility for ticker_to_check

    # what did we compute previously?
    σ_ii_previous = parameters_df[i, :volatility];
    
    # compute the difference -
    difference = abs(σ_ii - σ_ii_previous);
    println("σ_ii = $(σ_ii), previous = $(σ_ii_previous), abs(difference) = $difference")
end

The second method is even cooler! Let's check that the diagonal elements of the covariance rate $\hat C$ are approximately equal to the square of the __implied volatility__ estimates from the market!

> __Implied Volatility:__ The [implied volatility](https://en.wikipedia.org/wiki/Implied_volatility) is the market's forecast of a likely movement in a security's price. In other words, it's the volatility "implied" by the market price of an option based on an option pricing model (e.g., the [Black-Scholes model](https://en.wikipedia.org/wiki/Black%E2%80%93Scholes_model) or [Binomial model](https://en.wikipedia.org/wiki/Binomial_options_pricing_model)). 

> __How can we use it?__ The implied volatility represents one-standard deviation movement in the price of the underlying security over the next year. Therefore, if we square the implied volatility, we get an estimate of the variance of the security's returns over the next year. This is exactly what the diagonal elements of the covariance matrix represent!

Let's pop out to the market, and look at the implied volatility for a few firms. The implied volatility is often equal to or larger than the historical volatility (the diagonal elements of the covariance matrix we just computed).

___

## Task 2: Analyze the Covariance Matrix
In this task, let's pick out some interesting entries in the covariance matrix and analyze them. In particular, let's pick two tickers and see if their covariance is positive or negative, and then plot their growth rates to see if the data supports our findings.

Let's start by picking two tickers from our dataset. We store the ticker symbols in the `ticker_1::String` and `ticker_2::String` variables:

In [37]:
ticker_1 = "AMD"; # select a ticker in our dataset
ticker_2 = "NVDA"; # select a second ticker in our dataset

Next, let's get the data from our covariance matrix for our two tickers, and look at some values.

In [ ]:
let
    i = findfirst(x-> x==ticker_1, list_of_tickers);
    j = findfirst(x-> x==ticker_2, list_of_tickers);
    covariance_ij = Σ̂_g[i,j];
    σᵢ = sqrt(Σ̂_g[i,i]);
    σⱼ = sqrt(Σ̂_g[j,j]);
    correlation_ij = covariance_ij / (σᵢ * σⱼ);


    println("Covariance between $ticker_1 and $ticker_2 is approximately $covariance_ij")
    println("Correlation between $ticker_1 and $ticker_2 is approximately $correlation_ij")
end

Let's plot the growth rates for our two tickers to see if the data supports our findings.

In [ ]:
let

    # initialize -
    i = findfirst(x-> x==ticker_1, list_of_tickers);
    j = findfirst(x-> x==ticker_2, list_of_tickers);
    G = log_growth_matrix(dataset, list_of_tickers); # growth rate; r = Δt*g
    r, c = size(G); # rows, columns
    X = Array{Float64, 2}(undef, r, 2); # initialize matrix to hold growth rates for two tickers

    # collect the data -
    for t ∈ 1:r
        X[t,1] = G[t,i]; # growth rate for ticker_1
        X[t,2] = G[t,j]; # growth rate for ticker_2
    end

    # make xy-line -
    # Create x=y line data
    min_val = min(minimum(X[:,1]), minimum(X[:,2]))
    max_val = max(maximum(X[:,1]), maximum(X[:,2]))
    xy_line = [min_val, max_val]

    # plot -
    scatter(X[:,1], X[:,2], xlabel="Growth rate g of $ticker_1 (1/yr)", ylabel="Growth rate g of $ticker_2 (1/yr)", label="2014 - 2024", 
        c=:white, msc=:navy)
    plot!(xy_line, xy_line, color=:red, linestyle=:dash, label="", lw=2)
    plot!(bg="gray95", background_color_outside="white", framestyle = :box, fg_legend = :transparent);
end

### Discussion Questions
Why do we care about covariance (and correlation)?
1. What does it mean if two assets have positive covariance? Negative covariance? 
2. Is it a good thing to have assets with negative covariance in your portfolio? Why or why not?

___

## Summary
In this example, we computed the empirical covariance matrix for a portfolio of equities using historical growth-rate data. 

> __Key Takeaways:__
> * __Covariance matrix computation__: The empirical growth covariance is computed directly from the centered growth-rate matrix; the GBM covariance rate is $\hat C=(\Delta t)\hat\Sigma_g$. It can be verified against built-in functions, historical volatility, and forward-looking implied-volatility estimates.
> * __Covariance analysis__: Positive covariance indicates assets that tend to move together, while negative covariance shows assets that move oppositely, which has implications for portfolio diversification.
> * __Visualization insights__: Plotting growth rates of asset pairs provides visual confirmation of their covariance relationships and helps in understanding market dynamics.

__TL;DR__ We need the covariance for our simulations, but it also gives us some principled way to select tickers (may not want highly correlated tickers in our portfolio)

___


## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___